# mse-reconstruction-loss — worked example 2: Find the k samples with the highest per-sample reconstruction MSE

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mse-reconstruction-loss`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Computing per-sample reconstruction loss (by reducing only the spatial and channel dimensions, not the batch dimension) lets you identify which inputs are hardest for the model to reconstruct. This is a powerful diagnostic tool: sorting by per-sample loss and inspecting the worst-k examples often reveals systematic failure modes such as unusual textures or out-of-distribution images.

## Worked solution

**Step 1 — compute per-element MSE.** `F.mse_loss(pred, target, reduction='none')` gives shape `(B, C, H, W)`.

**Step 2 — reduce to per-sample scalar.** Average over all axes except batch: `.mean(dim=[1, 2, 3])` gives shape `(B,)`. Each entry is the mean squared error for one sample in the batch.

**Step 3 — find top-k indices.** `torch.topk(per_sample, k, largest=True)` returns the k largest values and their indices in descending order.

**Step 4 — return the indices and their losses.** The caller can use the indices to retrieve the original images for visualisation.

**Interpretation.** A high per-sample MSE means the model struggles with that specific input. In a VAE, this often corresponds to unusual samples that fall in low-density regions of the training distribution.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

def worst_k_samples(pred: torch.Tensor, target: torch.Tensor, k: int):
    """
    Return (indices, losses) of the k samples with the highest per-sample MSE.
    pred, target: (B, C, H, W)
    Returns:
      indices: LongTensor of shape (k,)
      losses:  FloatTensor of shape (k,), sorted descending
    """
    per_elem   = F.mse_loss(pred, target, reduction='none')  # (B, C, H, W)
    per_sample = per_elem.mean(dim=[1, 2, 3])                # (B,)
    topk_vals, topk_idx = torch.topk(per_sample, k, largest=True)
    return topk_idx, topk_vals

# Exercise on a batch with one deliberately bad reconstruction.
torch.manual_seed(3)
B, C, H, W = 8, 1, 14, 14
target = torch.rand(B, C, H, W)

# Most samples reconstruct well (small noise), but index 5 is terrible.
pred = target + 0.05 * torch.randn(B, C, H, W)
pred[5] = torch.randn(C, H, W)  # completely wrong reconstruction

idx, losses = worst_k_samples(pred, target, k=3)
print("Top-3 worst sample indices:", idx.tolist())
print("Top-3 losses:             ", [f"{v:.4f}" for v in losses.tolist()])
assert 5 in idx.tolist(), "Sample 5 should be in worst-3"
print("Worst-sample detection passed!")